In [2]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, hamming_loss, f1_score
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.multioutput import ClassifierChain

import pickle
import time 
import pandas as pd

## Load and Preprocess Data

In [2]:
# Load the data
df = pd.read_csv("./datasets/bemo_prompts_dataset_v2_gemini.csv")

In [3]:
# Show 5 random rows
df.sample(5)

,prompt,general,smart_home,todo,mail,others
547,"I need to know who won last night's game, open...",1,1,1,1,0
675,"What's a good book to read, and how do I sched...",1,0,1,0,0
281,Do you understand love?,0,0,0,0,1
1905,Engage the security alarm system.,0,1,0,0,0
2549,Could you provide a fun fact and then see to l...,1,1,0,0,0


In [20]:
# Split the data into X and Y
x = df["prompt"]
Y = df.drop(columns=["prompt"])
labels = Y.columns

In [10]:
print(x.shape, Y.shape)

(2568,) (2568, 5)


In [12]:
# Train-test split
x_train, x_test, Y_train, Y_test = train_test_split(x, Y, test_size=0.3, random_state=42, shuffle=True)

### Vanilla Classifier Chain with Logistic Regression

In [26]:
# Initialize the vectorizer
vectorizer = TfidfVectorizer()

# Initialize the classifier chain
classifier = ClassifierChain(LogisticRegression(), verbose=True)

In [27]:
# Create the pipeline

pipeline = Pipeline([
    ('vectorizer', vectorizer),
    ('classifier', classifier)
])

In [28]:
# Train the model
start = time.time()
pipeline.fit(x_train, Y_train)
end = time.time()

print(f"Training time: {end-start}")

[Chain] ................... (1 of 5) Processing order 0, total=   0.0s
[Chain] ................... (2 of 5) Processing order 1, total=   0.0s
[Chain] ................... (3 of 5) Processing order 2, total=   0.0s
[Chain] ................... (4 of 5) Processing order 3, total=   0.0s
[Chain] ................... (5 of 5) Processing order 4, total=   0.0s
Training time: 0.07637214660644531


In [29]:
# Predict
Y_pred = pipeline.predict(x_test)

In [30]:
# Evaluation
print("Accuracy:", accuracy_score(Y_test, Y_pred))
print("Hamming Loss:", hamming_loss(Y_test, Y_pred))
print("F1 Score (micro):", f1_score(Y_test, Y_pred, average='micro'))

Accuracy: 0.9182879377431906
Hamming Loss: 0.021011673151750974
F1 Score (micro): 0.9720013826477705


In [31]:
# Test with a new example
prompt = input("Enter a prompt: ")
new_prompt = [prompt]
start = time.time()
new_prediction = pipeline.predict(new_prompt)
end = time.time()

In [32]:
# Get the predicted labels
predicted_labels = [labels[i] for i, val in enumerate(new_prediction[0]) if val == 1]
print("\nInput:", new_prompt[0])
print("Predicted Tasks:", predicted_labels)
print("Time taken:", end-start)


Input: i love you
Predicted Tasks: ['others']
Time taken: 0.005205869674682617


In [36]:
# Save the model
with open("./models/TC_Pipeline_LR_v1.pkl", "wb") as f:
    pickle.dump(pipeline, f)

In [25]:
# Save the labels
with open("./models/labels.pkl", "wb") as f:
    pickle.dump(labels, f)

In [37]:
# Test the saved model
with open("./models/TC_Pipeline_LR_v1.pkl", "rb") as f:
    loaded_pipeline = pickle.load(f)
    
with open("./models/labels.pkl", "rb") as f:
    labels = pickle.load(f)
    
new_prediction = loaded_pipeline.predict(new_prompt)
predicted_labels = [labels[i] for i, val in enumerate(new_prediction[0]) if val == 1]
print("\nInput:", new_prompt[0])
print("Predicted Tasks:", predicted_labels)


Input: i love you
Predicted Tasks: ['others']


### Vanilla Classifier Chain with XGBoost

In [45]:
# Initialize the vectorizer
vectorizer = TfidfVectorizer()

# Initialize the classifier chain
classifier = ClassifierChain(GradientBoostingClassifier(), verbose=True)

In [46]:
# Create the pipeline

pipeline = Pipeline([
    ('vectorizer', vectorizer),
    ('classifier', classifier)
])

In [47]:
# Train the model
start = time.time()
pipeline.fit(x_train, Y_train)
end = time.time()
print(f"Training time: {end-start}")

[Chain] ................... (1 of 5) Processing order 0, total=   0.6s
[Chain] ................... (2 of 5) Processing order 1, total=   0.7s
[Chain] ................... (3 of 5) Processing order 2, total=   0.7s
[Chain] ................... (4 of 5) Processing order 3, total=   0.7s
[Chain] ................... (5 of 5) Processing order 4, total=   0.7s
Training time: 3.4161674976348877


In [48]:
# Predict
Y_pred = pipeline.predict(x_test)

In [49]:
# Evaluation
print("Accuracy:", accuracy_score(Y_test, Y_pred))
print("Hamming Loss:", hamming_loss(Y_test, Y_pred))
print("F1 Score (micro):", f1_score(Y_test, Y_pred, average='micro'))

Accuracy: 0.9182879377431906
Hamming Loss: 0.019974059662775615
F1 Score (micro): 0.9733656174334141


In [50]:
# Test with a new example
prompt = input("Enter a prompt: ")
new_prompt = [prompt]
start = time.time()
new_prediction = pipeline.predict(new_prompt)
end = time.time()

In [51]:
# Get the predicted labels
predicted_labels = [labels[i] for i, val in enumerate(new_prediction[0]) if val == 1]
print("\nInput:", new_prompt[0])
print("Predicted Tasks:", predicted_labels)
print("Time taken:", end-start)


Input: I love AI
Predicted Tasks: ['others']
Time taken: 0.009556770324707031


In [52]:
# Save the model
with open("./models/TC_Pipeline_GB_v1.pkl", "wb") as f:
    pickle.dump(pipeline, f)

### Classifier Chain with Logistic Regression and Grid Search for Hyperparameter Tuning

In [63]:
# Initialize the vectorizer
vectorizer = TfidfVectorizer()

# Initialize the classifier chain
classifier = ClassifierChain(LogisticRegression())

In [64]:
# Create the pipeline

pipeline = Pipeline([
    ('vectorizer', vectorizer),
    ('classifier', classifier)
])

In [65]:
# Define hyperparameter grid
param_grid = {
    'classifier__base_estimator__C': [0.01, 0.1, 1, 10],  # Regularization strength
    'classifier__base_estimator__penalty': ['l1', 'l2', 'elasticnet'],  # Regularization type
    'classifier__base_estimator__solver': ['liblinear', 'saga'],  # Solver (some only support specific penalties)
    'classifier__base_estimator__max_iter': [100, 200, 500],  # Number of iterations
    'classifier__base_estimator__tol': [1e-4, 1e-3, 1e-2],  # Stopping criteria tolerance
    'classifier__base_estimator__class_weight': [None, 'balanced'],  # Handle imbalanced data
    'vectorizer__max_features': [100, 500, 1000],  # Number of TF-IDF features
    'vectorizer__ngram_range': [(1, 1), (1, 2)]  # Use unigrams or bigrams
}

In [66]:
# Initialize the grid search
grid_search = GridSearchCV(pipeline, param_grid, cv=3, scoring='accuracy', verbose=2, n_jobs=-1)

In [ ]:
# Train the model
start = time.time()
grid_search.fit(x_train, Y_train)
end = time.time()
print(f"Training time: {end-start}")

In [68]:
# Print best parameters and best score
print("Best Parameters:", grid_search.best_params_)
print("Best Score:", grid_search.best_score_)

Best Parameters: {'classifier__base_estimator__C': 10, 'classifier__base_estimator__class_weight': 'balanced', 'classifier__base_estimator__max_iter': 500, 'classifier__base_estimator__penalty': 'l1', 'classifier__base_estimator__solver': 'saga', 'classifier__base_estimator__tol': 0.0001, 'vectorizer__max_features': 1000, 'vectorizer__ngram_range': (1, 2)}
Best Score: 0.9571508069003896


In [69]:
# Get the best model
best_pipeline = grid_search.best_estimator_

In [70]:
# Predict
Y_pred = best_pipeline.predict(x_test)

In [71]:
# Evaluation
print("Accuracy:", accuracy_score(Y_test, Y_pred))
print("Hamming Loss:", hamming_loss(Y_test, Y_pred))
print("F1 Score (micro):", f1_score(Y_test, Y_pred, average='micro'))

Accuracy: 0.9740596627756161
Hamming Loss: 0.007263294422827497
F1 Score (micro): 0.9904109589041096


In [76]:
# Test with a new example
prompt = input("Enter a prompt: ")
new_prompt = [prompt]
start = time.time()
new_prediction = best_pipeline.predict(new_prompt)
end = time.time()

In [77]:
# Get the predicted labels
predicted_labels = [labels[i] for i, val in enumerate(new_prediction[0]) if val == 1]
print("\nInput:", new_prompt[0])
print("Predicted Tasks:", predicted_labels)
print("Time taken:", end-start)


Input: remind me that i have meeting at 5 and send an email to ali reminding him too
Predicted Tasks: ['todo', 'mail']
Time taken: 0.00568079948425293


In [78]:
# Save the best model
with open("./models/TC_Pipeline_LR_v2.pkl", "wb") as f:
    pickle.dump(best_pipeline, f)